# Deployment Strategies

Notebook for **Chapter 8** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 Shadow Deployment, Blue/Green and Rollback

## 8.2 Canary Releases with Automatic Rollback

In [ ]:
def canary(true_err_new, seed, stages=(0.05, 0.25, 0.5, 1.0),
           per_stage=2000, base_err=0.02, margin=0.015):
    rng = random.Random(seed)
    for frac in stages:
        n = int(per_stage * frac)
        errs = sum(rng.random() < true_err_new for _ in range(n))
        rate = errs / n
        if rate > base_err + margin:
            return f"ROLLBACK at {int(frac*100)}% traffic (observed {rate:.3f}, limit {base_err+margin:.3f}, {n} reqs)"
    return "PROMOTED to 100%"

## 8.3 A/B Tests of Models

## 8.4 Multi-Armed Bandits and Choosing a Strategy

## 8.5 Release Math Workshop: Splits, Paired Tests, Sequential Canaries

The chapter so far gave you the strategies. This section gives you the arithmetic that makes each strategy trustworthy: how to check the traffic split is honest, how to compare a shadow model, how to decide a canary faster than a fixed-size test, and how to turn all of it into a pipeline. All outputs are real runs (scipy and plain Python); the Kubernetes parts are marked not run here.

In [ ]:
from scipy.stats import chisquare
# Sample ratio mismatch (SRM): we asked for a 90/10 canary split. Did we get it?
for n_old, n_new in ((9_412, 1_088), (9_700, 800)):
    total = n_old + n_new
    stat, p = chisquare([n_old, n_new], [total * 0.9, total * 0.1])
    print(n_old, n_new, "chi2 =", round(stat, 2), " p =", f"{p:.2e}")

9412 1088 chi2 = 1.53  p = 2.16e-01
9700 800 chi2 = 66.14  p = 4.21e-16


In [ ]:
import math
# Shadow test: both models score the same 2,000 requests; labels arrive later.
# b = old right & new wrong, c = old wrong & new right. Only these disagreements carry information.
b, c = 58, 91
# McNemar test (exact binomial on the b + c discordant cases)
from scipy.stats import binomtest
res = binomtest(c, b + c, 0.5)
print("new better on", c, "cases, worse on", b, "cases")
print("p-value:", round(res.pvalue, 4))
print("net gain:", c - b, "requests =", f"{(c - b) / 2000:.2%} accuracy")

new better on 91 cases, worse on 58 cases
p-value: 0.0085
net gain: 33 requests = 1.65% accuracy


In [ ]:
import math
# Sequential canary (SPRT): decide as soon as the evidence is strong enough.
# H0: new model error rate = 2.0% (as good as old).  H1: 3.0% (a real regression).
p0, p1 = 0.02, 0.03
alpha, beta = 0.05, 0.10               # false rollback, missed regression
upper = math.log((1 - beta) / alpha)   # cross it: roll back
lower = math.log(beta / (1 - alpha))   # cross it: promote
llr_err  = math.log(p1 / p0)
llr_ok   = math.log((1 - p1) / (1 - p0))
print("rollback above", round(upper, 2), " promote below", round(lower, 2))
print("each error adds", round(llr_err, 3), " each success adds", round(llr_ok, 4))
import random
def run(true_p, seed):
    rnd = random.Random(seed); s = 0.0
    for n in range(1, 100001):
        s += llr_err if rnd.random() < true_p else llr_ok
        if s >= upper: return "rollback", n
        if s <= lower: return "promote", n
    return "undecided", n
for true_p in (0.02, 0.03):
    res = [run(true_p, s) for s in range(1000)]
    rb = sum(r[0] == "rollback" for r in res) / 1000
    avg = sum(r[1] for r in res) / 1000
    print(f"true error {true_p:.0%}: rolled back {rb:.1%} of runs, average {avg:.0f} requests to decide")

rollback above 2.89  promote below -2.25
each error adds 0.405  each success adds -0.0103
true error 2%: rolled back 4.7% of runs, average 1021 requests to decide
true error 3%: rolled back 90.0% of runs, average 1128 requests to decide


In [ ]:
from scipy.stats import norm
import math
# Minimum detectable effect: with n users per arm, base rate 10%, what lift can we see?
# (alpha 0.05 two-sided, power 80%)
za, zb = norm.ppf(0.975), norm.ppf(0.80)
p = 0.10
print("users/arm   smallest detectable lift (points)")
for n in (1_000, 5_000, 20_000, 100_000):
    mde = (za + zb) * math.sqrt(2 * p * (1 - p) / n)
    print(f"{n:9,d}   {mde*100:5.2f}")

users/arm   smallest detectable lift (points)
    1,000    3.76
    5,000    1.68
   20,000    0.84
  100,000    0.38


In [ ]:
# Guardrails: ship only if the main metric improves AND no guardrail breaks.
results = {   # metric: (old, new, worst allowed change)
    "conversion":      (0.100, 0.107, None),
    "p99 latency ms":  (180,   205,   +30),
    "error rate":      (0.004, 0.004, +0.001),
    "revenue per user": (4.10, 4.05,  -0.10),
}
ok = True
for name, (old, new, limit) in results.items():
    if limit is None:
        print(f"{name}: {old} -> {new}  (target metric)"); continue
    delta = new - old
    breach = delta > limit if limit > 0 else delta < limit
    ok &= not breach
    print(f"{name}: change {delta:+.3f}, limit {limit:+}, {'BREACH' if breach else 'ok'}")
print("decision:", "promote" if ok else "hold")

conversion: 0.1 -> 0.107  (target metric)
p99 latency ms: change +25.000, limit +30, ok
error rate: change +0.000, limit +0.001, ok
revenue per user: change -0.050, limit -0.1, ok
decision: promote


## Chapter recap

## Practice set

**1.** (Easy) A canary sees 40 errors in 2,000 requests. The limit is 3.5%. Pass or roll back?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch08.html